In [1]:
# Import necessary libraries
import torch                          # Main PyTorch library
import torch.nn as nn                 # For building neural network layers
import torch.optim as optim           # For optimization algorithms
from torchvision import datasets, transforms   # For loading MNIST dataset
from torch.utils.data import DataLoader        # For batching dataset

# Check if GPU is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------
# Step 1: Define the Variational Autoencoder Model
# ------------------------------------------------------------

class VAE(nn.Module):

    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        """
        input_dim = size of input image (28x28 = 784)
        hidden_dim = number of neurons in hidden layer
        latent_dim = dimension of latent vector z
        """
        super(VAE, self).__init__()

        # ---------------- Encoder ----------------
        # This layer compresses the input image into a hidden representation
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        # These two layers produce parameters of latent distribution
        # Mean (μ)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)

        # Log variance (log σ²)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # ---------------- Decoder ----------------
        # Converts latent vector back to hidden representation
        self.fc2 = nn.Linear(latent_dim, hidden_dim)

        # Reconstructs the image from hidden representation
        self.fc3 = nn.Linear(hidden_dim, input_dim)

    # ------------------------------------------------------------
    # Encoder function
    # ------------------------------------------------------------
    def encode(self, x):
        """
        Converts input image into latent distribution parameters
        """
        h = torch.relu(self.fc1(x))     # Apply ReLU activation
        mu = self.fc_mu(h)              # Calculate mean μ
        logvar = self.fc_logvar(h)      # Calculate log variance log(σ²)

        return mu, logvar


    # ------------------------------------------------------------
    # Reparameterization Trick
    # ------------------------------------------------------------
    def reparameterize(self, mu, logvar):
        """
        Samples latent vector z using:
        z = μ + σ * ε
        where ε ~ N(0,1)
        """
        std = torch.exp(0.5 * logvar)   # Convert log variance to standard deviation
        eps = torch.randn_like(std)     # Generate random noise ε
        z = mu + eps * std              # Apply reparameterization trick

        return z


    # ------------------------------------------------------------
    # Decoder function
    # ------------------------------------------------------------
    def decode(self, z):
        """
        Reconstructs the original image from latent vector z
        """
        h = torch.relu(self.fc2(z))     # Hidden layer
        x_reconstructed = torch.sigmoid(self.fc3(h))  # Output layer

        return x_reconstructed


    # ------------------------------------------------------------
    # Forward pass
    # ------------------------------------------------------------
    def forward(self, x):
        """
        Defines the full pipeline of the VAE
        """
        mu, logvar = self.encode(x)     # Step 1: Encoder
        z = self.reparameterize(mu, logvar)  # Step 2: Sampling
        x_reconstructed = self.decode(z)     # Step 3: Decoder

        return x_reconstructed, mu, logvar



# ------------------------------------------------------------
# Step 2: Load MNIST Dataset
# ------------------------------------------------------------

transform = transforms.ToTensor()   # Convert images to tensor format

# Download training dataset
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Create data loader for batching
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)



# ------------------------------------------------------------
# Step 3: Initialize Model and Optimizer
# ------------------------------------------------------------

model = VAE().to(device)        # Move model to GPU/CPU
optimizer = optim.Adam(model.parameters(), lr=1e-3)   # Adam optimizer



# ------------------------------------------------------------
# Step 4: Define Loss Function
# ------------------------------------------------------------

def loss_function(reconstructed_x, x, mu, logvar):
    """
    Total VAE loss = Reconstruction Loss + KL Divergence
    """

    # Flatten input image
    x = x.view(-1, 784)

    # Reconstruction loss (Binary Cross Entropy)
    recon_loss = nn.functional.binary_cross_entropy(
        reconstructed_x, x, reduction='sum'
    )

    # KL Divergence loss
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + kl_loss, recon_loss, kl_loss



# ------------------------------------------------------------
# Step 5: Training Loop
# ------------------------------------------------------------

num_epochs = 10   # Number of training iterations

for epoch in range(num_epochs):

    total_loss = 0

    for batch_idx, (data, _) in enumerate(train_loader):

        data = data.to(device)

        # Flatten image from (28x28) to 784
        data = data.view(-1, 784)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass through VAE
        reconstructed, mu, logvar = model(data)

        # Calculate loss
        loss, recon_loss, kl_loss = loss_function(reconstructed, data, mu, logvar)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        total_loss += loss.item()

    # Print loss after each epoch
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Total Loss: {total_loss:.2f}")
    print(f"Reconstruction Loss: {recon_loss.item():.2f}")
    print(f"KL Divergence Loss: {kl_loss.item():.2f}")
    print("-----------------------------------")



# ------------------------------------------------------------
# Step 6: Testing Reconstruction
# ------------------------------------------------------------

with torch.no_grad():

    # Get one batch of images
    sample, _ = next(iter(train_loader))
    sample = sample.to(device)
    sample = sample.view(-1, 784)

    # Pass through VAE
    reconstructed, _, _ = model(sample)

    print("Reconstruction completed")

100%|██████████| 9.91M/9.91M [00:00<00:00, 56.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.72MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.7MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.32MB/s]


Epoch 1/10
Total Loss: 9891160.83
Reconstruction Loss: 10905.56
KL Divergence Loss: 1929.34
-----------------------------------
Epoch 2/10
Total Loss: 7278849.11
Reconstruction Loss: 9134.45
KL Divergence Loss: 2323.67
-----------------------------------
Epoch 3/10
Total Loss: 6865271.26
Reconstruction Loss: 8813.19
KL Divergence Loss: 2210.60
-----------------------------------
Epoch 4/10
Total Loss: 6694351.43
Reconstruction Loss: 7826.67
KL Divergence Loss: 2232.73
-----------------------------------
Epoch 5/10
Total Loss: 6592704.71
Reconstruction Loss: 8137.10
KL Divergence Loss: 2231.13
-----------------------------------
Epoch 6/10
Total Loss: 6520706.69
Reconstruction Loss: 7620.94
KL Divergence Loss: 2378.41
-----------------------------------
Epoch 7/10
Total Loss: 6470243.65
Reconstruction Loss: 7771.59
KL Divergence Loss: 2392.15
-----------------------------------
Epoch 8/10
Total Loss: 6431013.84
Reconstruction Loss: 7678.02
KL Divergence Loss: 2377.60
-------------------